In [10]:
import pandas as pd

news_df = pd.read_csv(r"C:\sentiment_analysis\raw_analyst_ratings.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\AAPL.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\AMZN.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\GOOG.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\META.csv")
stock_df = pd.read_csv(r"C:\sentiment_analysis\Data\NVDA.csv")
print(news_df.columns)
news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce")
news_df = news_df.dropna(subset=['date'])
news_df['date'] = pd.to_datetime(news_df['date'], utc=True).dt.tz_convert(None)
print(stock_df.columns)
stock_df["Date"] = pd.to_datetime(stock_df["Date"]).dt.tz_localize(None)

stock_df = stock_df.sort_values('Date')

Index(['Unnamed: 0', 'headline', 'url', 'publisher', 'date', 'stock'], dtype='str')
Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')


check the output 

In [11]:
print(stock_df[['Date', 'Close', 'daily_return']].head())

KeyError: "['daily_return'] not in index"

In [ ]:
news_df['trading_day'] = news_df['date'].apply(
    lambda x: align_to_trading_day(x, trading_days)
)


stock_df = stock_df.sort_values('Date')
trading_days = stock_df['Date'].unique()
trading_days = pd.to_datetime(trading_days).tz_localize(None)

def align_to_trading_day(ts, trading_days):
    if pd.isna(ts):
        return None
    ts = ts.normalize()
    idx = trading_days.searchsorted(ts)
    if idx < len(trading_days):
        return trading_days[idx]
    return None

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

news_df['sentiment_score'] = news_df['headline'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

print(news_df[['date', 'trading_day']].head())

daily_sentiment = (
    news_df.groupby('trading_day')['sentiment_score']
    .mean()
    .reset_index()
)

                 date trading_day
0 2020-06-05 14:30:54  2020-06-05
1 2020-06-03 14:45:20  2020-06-03
2 2020-05-26 08:30:07  2020-05-26
3 2020-05-22 16:45:06  2020-05-22
4 2020-05-22 15:38:59  2020-05-22


Merge with Daily Stock Returns

In [ ]:
merged = pd.merge(
    stock_df[['Date', 'daily_return']],
    daily_sentiment,
    left_on='Date',
    right_on='trading_day',
    how='left'
)

fill missing sentiment values

In [ ]:
merged['sentiment_score'] = merged['sentiment_score'].fillna(0)

computing Pearson correlation

In [ ]:
from scipy.stats import pearsonr

corr, p = pearsonr(
    merged['sentiment_score'].fillna(0),
    merged['daily_return'].fillna(0)
)

print("Pearson Correlation:", corr)
print("P-value:", p)

Correlation: 0.01817888629245388
P-value: 0.26420653148556333
